# 01. NDC topic model

The government side of the analysis: 195 translated NDCs become 2,762 text chunks,
which are embedded with ClimateBERT and clustered with BERTopic into the 38 topics
(T1-T38) that the rest of the paper is built on.

## Read this before running

**Sections 2 and 3 are frozen and should not be re-run.** UMAP and HDBSCAN results
depend on the numba and BLAS builds underneath them, so the same seed on a different
machine can produce a different clustering, and therefore a different topic numbering.
T1-T38, the six thematic groups G1-G6, Supplementary Table S1 and Figure 1 all refer
to the numbering produced by the original run, so re-clustering would invalidate them.

The fitted model and its output are kept in `output/` and loaded by sections 4 and 5:

| File | What it is |
|---|---|
| `output/Bert_topic_model` | the fitted BERTopic model |
| `output/GA_topics.csv` | 2,762 chunks x 38 topic probabilities, the GA weights |
| `assets/topic_keywords.json` | top keywords per topic, read out of the model |
| `assets/topic_sentences.json` | the 38 hand-written topic sentences |

Sections 2 and 3 keep the code that produced these, so the method is documented and
auditable. Set `RERUN_TOPIC_MODEL = True` only if you intend to rebuild the topic
numbering from scratch and renumber the manuscript with it.

## Dependencies

Sections 1, 4 and 5 need only `pandas`, `nltk` and `gensim`. Sections 2 and 3
additionally need `torch`, `sentence_transformers`, `bertopic`, `umap-learn` and
`hdbscan`, pinned to `numpy==1.26.4` (BERTopic 0.16.4 does not support NumPy 2), and
a GPU.

```
pip install pandas nltk==3.9.1 gensim                       # sections 1, 4, 5
pip install numpy==1.26.4 bertopic==0.16.4 \
            sentence_transformers==3.4.1 umap-learn hdbscan torch   # sections 2, 3
```

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # the modules the notebooks import live in src/


In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.stem import WordNetLemmatizer

from gensim.models import Phrases
from gensim.models.phrases import Phraser

import config as cfg

# Leave this False. See "Read this before running" above.
RERUN_TOPIC_MODEL = False

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print("NDC texts :", cfg.NDC_TXT_DIR)
print("Output    :", cfg.OUTPUT_DIR)

## 1. Process NDC documents

Each Party's most recent NDC published before 2023, translated into English where the
original was in another language. The EU submits one NDC covering its 27 member
states, so it appears once.

In [ ]:
### Combine all NDCs into one table

dirs = [x for x in os.listdir(cfg.NDC_TXT_DIR) if x.split('.')[-1] == 'txt']

country_lst = []
docs = []
for d in sorted(dirs):
    full_path = os.path.join(cfg.NDC_TXT_DIR, d)
    # "Colombia_Spanish.txt" -> "Colombia"
    country_name = d.split('.')[0].split('_')[0]
    with open(full_path, 'r', encoding='utf-8', errors='replace') as file:
        docs.append(file.read())
        country_lst.append(country_name)

df = pd.DataFrame({'Country': country_lst, 'Document': docs})
print("Number of NDC documents:", len(df))
df.head()

In [ ]:
### Text cleaning, define a preprocessing function

def clean_text(doc, stopwords_set):
    """
    Cleans a single text document:
    1) Lowercase
    2) Remove punctuation, digits, and non-Latin characters
    3) Tokenize
    4) Remove stop words
    5) Lemmatize
    Returns a list of cleaned tokens.

    Lemmatization happens here, before collocations are formed in the bigram step
    below, so that "emissions reduction" and "emission reductions" collapse onto the
    same bigram rather than competing with each other.
    """

    # 1) Lowercase
    doc = doc.lower()

    # 2) Remove punctuation, digits, and non-Latin symbols
    # Keep only letters (a-z) and whitespace
    doc = re.sub(r'[^a-z\s]', ' ', doc)

    # 3) Tokenize on whitespace
    tokens = doc.split()

    # 4) Remove stop words (including country and capital-city names)
    tokens = [word for word in tokens if word not in stopwords_set]

    # 5) Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    return tokens

In [ ]:
### Build the stop words set and clean every document

stopwords_set = cfg.stopwords_set()
print("Stop words:", len(stopwords_set))

df["Cleaned_Doc"] = df["Document"].apply(lambda x: clean_text(x, stopwords_set))
df[["Country", "Cleaned_Doc"]].head()

In [ ]:
### Remove rare words (appear fewer than twice overall)

# 1) Flatten all tokens
all_tokens = [token for tokens_list in df["Cleaned_Doc"] for token in tokens_list]

# 2) Count frequency
freq = Counter(all_tokens)

# 3) Identify rare words
rare_words = {word for word, count in freq.items() if count < cfg.MIN_WORD_FREQ}

# 4) Filter out rare words
df["Cleaned_Doc"] = df["Cleaned_Doc"].apply(
    lambda tokens: [token for token in tokens if token not in rare_words]
)

print("Vocabulary before / after:", len(freq), "/", len(freq) - len(rare_words))

In [ ]:
### Form bigrams, e.g. climate_change, ghg_emission, mitigation_action

token_lists = df["Cleaned_Doc"].tolist()

bigram_model = Phrases(token_lists,
                       min_count=cfg.BIGRAM_MIN_COUNT,
                       threshold=cfg.BIGRAM_THRESHOLD)
bigram_phraser = Phraser(bigram_model)

df["Document_processed"] = df["Cleaned_Doc"].apply(lambda x: bigram_phraser[x])

bigrams = df['Document_processed'].map(lambda x: [w for w in x if "_" in w])
bigram_counts = Counter(w for row in bigrams for w in row)
print("Distinct bigrams:", len(bigram_counts))
pd.Series(bigram_counts).sort_values(ascending=False).head(10)

In [ ]:
### Split the text into 500-word chunks (BERT limit: 512 tokens)

def split_into_chunks(text, max_words=cfg.CHUNK_WORDS):
    """
    Splits the text into chunks of up to max_words each,
    helps avoid exceeding model sequence length in embeddings.
    """
    words = text.split()
    chunks = [
        ' '.join(words[i:i + max_words])
        for i in range(0, len(words), max_words)
    ]
    return chunks


df['Document_string'] = df['Document_processed'].apply(lambda tokens: ' '.join(tokens))
df['Document_chunks'] = df['Document_string'].apply(split_into_chunks)

# Expand each list of chunks into separate rows
df_split = df.explode('Document_chunks', ignore_index=True)
df_split = df_split[['Country', 'Document_chunks']].rename(
    columns={'Document_chunks': 'Document_processed'})

document_chunks = df_split['Document_processed'].tolist()
print("Number of chunks:", len(document_chunks), " (2,762 in the published run)")

df_split.to_pickle(os.path.join(cfg.OUTPUT_DIR, 'ndc_processed_chunks.pkl'))
df_split.head()

## 2. Embedding with ClimateBERT — frozen

ClimateBERT is a DistilRoBERTa fine-tuned on climate text, so it separates
climate-specific vocabulary better than a general-purpose encoder.

Each 500-word chunk is embedded in 20-word pieces (mean pooling over tokens), then
max-pooled across pieces into one 768-dimensional vector. Max pooling keeps the
strongest signal from any part of the chunk, rather than averaging several distinct
policy statements into an indistinct middle.

Guarded by `RERUN_TOPIC_MODEL`; the output lives in `output/ndc_embeddings.csv`.

In [ ]:
if RERUN_TOPIC_MODEL:
    import torch
    from tqdm import tqdm
    from sentence_transformers import models, SentenceTransformer
    from transformers import AutoTokenizer, AutoModel

    # 1) Load the tokenizer and model from Hugging Face
    tokenizer = AutoTokenizer.from_pretrained(cfg.EMBEDDING_MODEL)
    huggingface_model = AutoModel.from_pretrained(cfg.EMBEDDING_MODEL)

    # 2) Build a Sentence Transformers "Transformer" module.
    # The warning about the lost pooling weight layer does not matter here:
    # we only need token embeddings plus our own mean/max pooling.
    transformer = models.Transformer(model_name_or_path=cfg.EMBEDDING_MODEL,
                                     max_seq_length=512)
    transformer.auto_model = huggingface_model
    transformer.tokenizer = tokenizer

    # 3) Mean pooling over tokens
    pooling = models.Pooling(
        word_embedding_dimension=transformer.get_word_embedding_dimension(),
        pooling_mode_mean_tokens=True,
        pooling_mode_cls_token=False,
        pooling_mode_max_tokens=False
    )

    # 4) Assemble
    embedding_model = SentenceTransformer(modules=[transformer, pooling])

    def chunk_document(doc, chunk_size=cfg.EMBED_SUBCHUNK_WORDS):
        """Split a chunk (string) into lists of 20-word pieces."""
        tokens = doc.split()
        return [" ".join(tokens[i:i + chunk_size])
                for i in range(0, len(tokens), chunk_size)]

    all_doc_embeddings = []
    for i in tqdm(range(len(document_chunks)), desc='Embedding chunks', unit='chunk'):
        pieces = chunk_document(document_chunks[i])
        if not pieces:                       # edge case: empty chunk
            all_doc_embeddings.append(torch.zeros(768))
            continue
        piece_embeddings = embedding_model.encode(pieces, convert_to_tensor=True)
        # Max pooling across pieces -> one vector per chunk
        all_doc_embeddings.append(torch.max(piece_embeddings, dim=0).values)

    all_doc_embeddings = torch.stack(all_doc_embeddings).cpu().numpy()
    print("Shape of chunk embeddings:", all_doc_embeddings.shape)
    pd.DataFrame(all_doc_embeddings).to_csv(
        os.path.join(cfg.OUTPUT_DIR, 'ndc_embeddings.csv'), header=False, index=False)
else:
    print("Skipped: using the frozen model in output/Bert_topic_model")

## 3. BERTopic model — frozen

UMAP reduces the 768 dimensions to 10, HDBSCAN clusters the chunks, and BERTopic
labels the clusters with c-TF-IDF keywords. The number of topics was tuned from 5 to
55 and 38 gave the highest coherence (Supplementary Figure S1).

HDBSCAN leaves 45.2% of chunks in the outlier cluster (-1). That is not discarded
data: the analysis uses `probabilities_`, a soft distribution over the 38 topics for
every chunk, because a chunk of policy text usually touches several topics at once.
The hard label is never used.

In [ ]:
if RERUN_TOPIC_MODEL:
    import random
    import torch
    import umap
    from hdbscan import HDBSCAN
    from bertopic import BERTopic
    from sklearn.feature_extraction.text import CountVectorizer

    embeddings = pd.read_csv(os.path.join(cfg.OUTPUT_DIR, 'ndc_embeddings.csv'),
                             header=None).values

    # Fix the random state so the topic numbering is reproducible on this machine
    random.seed(cfg.SEED)
    np.random.seed(cfg.SEED)
    torch.manual_seed(cfg.SEED)

    umap_model = umap.UMAP(**cfg.UMAP_KWARGS)
    hdbscan_model = HDBSCAN(**cfg.HDBSCAN_KWARGS)

    # Units, URL fragments and document furniture that survive cleaning and would
    # otherwise dominate the topic keywords.
    vectorizer_model = CountVectorizer(stop_words=cfg.DOMAIN_STOPWORDS, min_df=2)

    # nr_topics=39 yields 38 topics plus the -1 outlier cluster
    topic_model = BERTopic(
        min_topic_size=cfg.MIN_TOPIC_SIZE,
        top_n_words=cfg.TOP_N_WORDS,
        nr_topics=cfg.N_TOPICS,
        calculate_probabilities=True,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model
    )
    topics, probabilities = topic_model.fit_transform(document_chunks, embeddings)
    print("Probabilities:", probabilities.shape)
    print(f"Outlier (-1) share: {(np.array(topics) == -1).mean():.1%}")

    topic_model.save(os.path.join(cfg.OUTPUT_DIR, 'Bert_topic_model'))

    # Normalise so each chunk's 38 topic weights sum to 1, then write GA
    probabilities = probabilities / probabilities.sum(axis=1)[:, None]
    ga = df_split[['Country', 'Document_processed']].copy()
    ga[cfg.ndc_columns()] = probabilities
    ga.to_csv(os.path.join(cfg.OUTPUT_DIR, 'GA_topics.csv'), index=False)
else:
    print("Skipped: using the frozen output/GA_topics.csv")

## 4. Country-level topic weights (GA)

Each chunk carries a vector of 38 probabilities rather than a single label. A Party's
weights are the mean across its chunks, which is GA in Eq. (1).

In [ ]:
ga = pd.read_csv(os.path.join(cfg.OUTPUT_DIR, 'GA_topics.csv'), low_memory=False)
sub_cols = cfg.ndc_columns()

print("Chunks:", len(ga), "  Parties:", ga.Country.nunique())
print("Each chunk's weights sum to:", round(float(ga[sub_cols].sum(axis=1).mean()), 6))

ga_country = ga.groupby('Country', as_index=False)[sub_cols].mean()
ga_country.to_csv(os.path.join(cfg.OUTPUT_DIR, 'GA_by_country.csv'), index=False)
ga_country.head()

In [ ]:
### Group the 38 topics into the six thematic groups G1-G6

for group, subtopics in cfg.TOPIC_GROUPS_NDC.items():
    cols = [f'Subtopic_{i}' for i in subtopics]
    ga_country[group] = ga_country[cols].sum(axis=1)

print("Mean NDC weight per thematic group")
for g in cfg.TOPIC_GROUPS_NDC:
    print(f"  {g}  {cfg.GROUP_NAMES[g]:<34s} {ga_country[g].mean():.2%}")

## 5. The 38 topics

BERTopic returns keyword lists. The one-line summaries below were written by hand
after reviewing the keywords and sample chunks for each topic, and they are what every
post is matched against in notebook 03. They are fixed in
`assets/topic_sentences.json`: T1-T38 and G1-G6 throughout the manuscript refer to
these, so they must not change between runs.

**T1-T38 are not BERTopic's own numbering.** The topics were renumbered by hand when
they were read and grouped, so T1 is BERTopic topic 1, T2 is BERTopic topic 0, T5 is
BERTopic topic 6, and so on. The permutation lives in
`MAP_TW_TOPIC_TO_NDC_SUBTOPIC` and must be applied before pulling keywords out of the
model, otherwise the keywords are attached to the wrong topic. `cfg.tw_topic_to_bertopic_index`
does this. The check at the end of the cell confirms the permutation is a bijection
and that it maps the post-side groups onto the NDC-side groups exactly.

The table written here is the in-text topic table Referee 1 asked for (R1.2).

In [ ]:
import json

with open(os.path.join(cfg.ASSETS_DIR, 'topic_keywords.json'), encoding='utf-8') as fh:
    keywords = json.load(fh)

topics_tbl = pd.DataFrame({
    'topic': [f'T{i}' for i in range(1, 39)],
    'sentence': cfg.TOPIC_SENTENCES,
})
topics_tbl['group'] = [
    next(g for g, ids in cfg.TOPIC_GROUPS_TW.items() if i in ids)
    for i in range(1, 39)
]
topics_tbl['group_name'] = topics_tbl.group.map(cfg.GROUP_NAMES)

# T1-T38 are a hand renumbering of BERTopic's 0..37, so go through the permutation
topics_tbl['bertopic_id'] = [cfg.tw_topic_to_bertopic_index(i) for i in range(1, 39)]
topics_tbl['keywords'] = [
    ', '.join(w for w, _ in keywords[str(b)][:10]) for b in topics_tbl.bertopic_id
]

topics_tbl.to_csv(os.path.join(cfg.OUTPUT_DIR, 'topic_table.csv'), index=False)

### Check the renumbering is internally consistent
assert sorted(topics_tbl.bertopic_id) == list(range(38)), "permutation is not a bijection"
for g, ids in cfg.TOPIC_GROUPS_TW.items():
    mapped = sorted(cfg.tw_topic_to_bertopic_index(t) for t in ids)
    assert mapped == sorted(cfg.TOPIC_GROUPS_NDC[g]), f"{g} groups disagree"
print("Renumbering check passed: bijection, and the six groups agree on both sides.")
print("Topics per group:", topics_tbl.group.value_counts().sort_index().to_dict())

topics_tbl[['topic', 'bertopic_id', 'group', 'sentence']]